# 💬 Урок 18 — LLM на практике (материалы преподавателя)

> 🎯 Цель: промптинг, классификация тональности, критическая оценка ответа.

Блок 2 работает офлайн. Блок 1 — Hugging Face (без ключа, скачивает модель). Блоки 3–4 — любой чат-LLM или API.

## Блок 1 · Тональность через готовую модель (Hugging Face, без ключа)

In [ ]:
!pip install transformers -q
from transformers import pipeline
sentiment = pipeline('sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment')
for о in ['Отличный сервис, всем советую!','Ужасно, больше не приду',
          'Ну очень порадовал этот сервис...']:
    print(о[:35], '->', sentiment(о)[0]['label'])

## Блок 2 · Классификатор словарём (принцип, офлайн)
Показывает идею — и его слабые места (сарказм, отрицание).

In [ ]:
позитив = {'отличный','супер','советую','нравится','быстро','вежливо','класс'}
негатив = {'ужасно','плохо','долго','грубо','обман','отвратительно'}
def тональность(текст):
    слова = текст.lower().replace('!','').replace('.','').split()
    p = sum(w in позитив for w in слова)
    n = sum(w in негатив for w in слова)
    return 'позитив' if p>n else 'негатив' if n>p else 'нейтрально'
for о in ['отличный сервис советую','ужасно и долго','ну очень порадовал этот сервис']:
    print(f'{о:40s} -> {тональность(о)}')

## Блок 3 · Промпт с ролью и примерами (few-shot)
Работает в любом чат-LLM или через API.

In [ ]:
prompt = '''Ты — строгий, но справедливый модератор отзывов.
Определи тональность: позитив / негатив / нейтрально.

Примеры:
Отзыв: "Всё быстро и вежливо" -> позитив
Отзыв: "Ждал час, никто не помог" -> негатив

Отзыв: "Ну очень порадовал этот сервис..." ->'''
print(prompt)

## Блок 4 · Бот с характером (системный промпт)
Системный промпт задаёт роль и стиль.

In [ ]:
system = 'Ты — дружелюбный кот-программист. Отвечай коротко, с юмором.'
вопрос = 'Что такое переменная?'
# Вставьте system + вопрос в чат-LLM (ChatGPT/Claude) или в API вашего провайдера.
print('SYSTEM:', system)
print('USER  :', вопрос)

## Блок 5 (для сильных) · Мини-RAG: ответ по своему тексту

In [ ]:
текст = 'Наш магазин работает с 9 до 21. Доставка бесплатно от 2000 руб.'
вопрос = 'До скольких работает магазин?'
куски = [s.strip() for s in текст.split('.') if s.strip()]
нужный = [s for s in куски if 'работает' in s][0]
print('Контекст:', нужный)
print('Вопрос  :', вопрос)
print('-> модель отвечает СТРОГО по контексту — меньше галлюцинаций')

---
**Итог.** Роль + примеры + цепочка мыслей делают ответ лучше. Но модель проверяем всегда: она ошибается и бывает предвзятой.